# Phase 11.8 — Recommendation Pipeline

Application orchestration: 11.2 → 11.3 → 11.4 → 11.5 → 11.6 → `RecommendationResponse`.

Not evaluation; not HTTP (Phase 13).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.recommendation.baseline_ranker import BaselineRecommendationRanker
from productiq.recommendation.catalog_embeddings import InMemoryProductEmbeddingProvider
from productiq.recommendation.config import RecommendationConfig
from productiq.recommendation.content_similarity_engine import ContentSimilarityEngine
from productiq.recommendation.contracts import RecommendationRequest, RecommendationType
from productiq.recommendation.generators import AttributeRecommendationCandidateGenerator
from productiq.recommendation.recommendation_pipeline import RecommendationPipeline
from productiq.recommendation.recommendation_product_context_provider import (
    InMemoryRecommendationProductContextProvider,
)
from productiq.recommendation.seed_product import InMemoryRecommendationCatalog
from productiq.representation.filtering import FilteringRepresentation
from productiq.representation.schema import ProductRepresentation
from productiq.retrieval.semantic import SemanticVector

In [ ]:
def product(product_id: str) -> ProductRepresentation:
    return ProductRepresentation(
        product_id=product_id,
        brand="nike",
        brand_normalized="nike",
        category_gender="Women",
        product_type="t_shirt",
        color=["black"],
    )


def filtering(product_id: str) -> FilteringRepresentation:
    return FilteringRepresentation(
        product_id=product_id,
        source="ajio",
        brand="nike",
        brand_normalized="nike",
        category_gender="Women",
        product_type="t_shirt",
        color_raw="black",
        color_is_coded=True,
        color=["black"],
        pattern=None,
        material=None,
        fit=None,
        sleeve=None,
        neckline=None,
        product_features=None,
        style_attributes=None,
        discount_price_inr=1000,
        original_price_inr=1200,
        price_anomaly=False,
    )


catalog = InMemoryRecommendationCatalog([product(f"P{i}") for i in range(1, 7)])
filtering_by_id = {f"P{i}": filtering(f"P{i}") for i in range(1, 7)}
embeddings = {
    f"P{i}": SemanticVector(values=tuple(1.0 if j == i % 3 else 0.0 for j in range(3)))
    for i in range(1, 7)
}
pipeline = RecommendationPipeline(
    catalog=catalog,
    generators=(AttributeRecommendationCandidateGenerator(catalog=catalog),),
    similarity_engine=ContentSimilarityEngine(
        embedding_provider=InMemoryProductEmbeddingProvider(embeddings)
    ),
    ranker=BaselineRecommendationRanker(),
    product_context_provider=InMemoryRecommendationProductContextProvider(
        products_by_id={p.product_id: p for p in catalog.list_catalog_products()},
        filtering_by_product_id=filtering_by_id,
    ),
    filtering_by_product_id=filtering_by_id,
)
request = RecommendationRequest(
    seed_product_id="P1",
    recommendation_type=RecommendationType.SIMILAR,
    top_k=3,
    config=RecommendationConfig(),
)
result = pipeline.recommend_with_metadata(request)
response = result.response
print("seed:", response.seed_product_id)
print("requested_top_k:", response.requested_top_k)
print("candidate_count:", result.execution.candidate_count)
print("ranked_count:", result.execution.ranked_count)
print("selected:", result.execution.selected_count)
for row in response.recommendations:
    print(row.rank, row.product_id, row.recommendation_score)